# Lab Assignment-9: Cognitive Computing (UCS420)
## Landing AI (Vision based Cognitive System)
**Name:** Arshpreet Singh
**Roll No:** 1024170170

**Tasks**
1. Cognitive Assistant for reading labels of a medicine bottle (OCR + information extraction + speech output)
2. Emotion-aware Cognitive Assistant using facial expression detection

Run on Google Colab (GPU optional). Each part is independent.

---
# PART 1: Cognitive Assistant for Reading Medicine Bottle Labels

**Pipeline:** Image -> Pre-processing (OpenCV) -> OCR (EasyOCR) -> Information extraction (regex / NLP) -> Safety checks (expiry) -> Spoken output (gTTS)

A dataset can be downloaded from the internet (e.g. Kaggle: *Medicine Labels / Pharmaceutical Drugs and Vitamins Synthetic Images*). If no dataset is available, the notebook generates a synthetic label so everything runs end to end.

In [ ]:
!pip -q install easyocr opencv-python-headless gTTS python-dateutil

In [ ]:
import os, re, glob
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont
from matplotlib import font_manager
from datetime import datetime
from dateutil import parser as dparser
import easyocr
from gtts import gTTS
from IPython.display import Audio, display
print('Libraries loaded')

### 1.1 Get an input image
Option A: put your own medicine label images in a folder `medicine_dataset/`.
Option B (default fallback): generate a synthetic medicine label.

In [ ]:
def make_sample_label(path='sample_label.png'):
    img = Image.new('RGB', (900, 500), 'white')
    d = ImageDraw.Draw(img)
    fp = font_manager.findfont('DejaVu Sans')
    big = ImageFont.truetype(fp, 48)
    mid = ImageFont.truetype(fp, 30)
    small = ImageFont.truetype(fp, 24)
    d.rectangle([10, 10, 890, 490], outline='black', width=4)
    d.text((40, 30), 'PARACETAMOL', fill='black', font=big)
    d.text((40, 100), 'Tablets IP 500 mg', fill='black', font=mid)
    d.text((40, 160), 'Dosage: 1 tablet every 6 hours after food', fill='black', font=small)
    d.text((40, 210), 'Batch No: BT2045A', fill='black', font=small)
    d.text((40, 260), 'Mfg Date: 03/2024', fill='black', font=small)
    d.text((40, 310), 'Exp Date: 02/2027', fill='black', font=small)
    d.text((40, 370), 'Store below 25 C. Keep out of reach of children.', fill='black', font=small)
    d.text((40, 420), 'Warning: Do not exceed 4 tablets in 24 hours.', fill='black', font=small)
    img.save(path)
    return path

DATASET_DIR = 'medicine_dataset'
image_paths = sorted(glob.glob(os.path.join(DATASET_DIR, '*.*')))
if not image_paths:
    print('No dataset folder found -> using a synthetic label')
    image_paths = [make_sample_label()]
print('Images to process:', image_paths[:5])

img = cv2.imread(image_paths[0])
plt.figure(figsize=(8, 5)); plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); plt.axis('off'); plt.title('Input image'); plt.show()

### 1.2 Pre-processing
Grayscale -> upscale -> denoise -> adaptive threshold improves OCR accuracy on curved / low contrast bottle labels.

In [ ]:
def preprocess(image_bgr):
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    gray = cv2.resize(gray, None, fx=1.5, fy=1.5, interpolation=cv2.INTER_CUBIC)
    gray = cv2.fastNlMeansDenoising(gray, None, 10, 7, 21)
    thr = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 31, 15)
    return gray, thr

gray, thr = preprocess(img)
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].imshow(gray, cmap='gray'); ax[0].set_title('Gray + denoised'); ax[0].axis('off')
ax[1].imshow(thr, cmap='gray'); ax[1].set_title('Adaptive threshold'); ax[1].axis('off')
plt.show()

### 1.3 Text recognition (OCR) with EasyOCR

In [ ]:
reader = easyocr.Reader(['en'], gpu=False)

def run_ocr(image_bgr, conf_thresh=0.3):
    gray, _ = preprocess(image_bgr)
    results = reader.readtext(gray)
    results = [r for r in results if r[2] >= conf_thresh]
    return results

results = run_ocr(img)
vis = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).copy()
scale = 1.5
for box, text, conf in results:
    pts = (np.array(box) / scale).astype(int)
    cv2.polylines(vis, [pts], True, (0, 200, 0), 2)
plt.figure(figsize=(9, 5)); plt.imshow(vis); plt.axis('off'); plt.title('Detected text regions'); plt.show()

full_text = '\n'.join([t for _, t, _ in results])
print(full_text)

### 1.4 Information extraction
Extract drug name, strength, dosage instructions, batch number, manufacture and expiry date, and warnings.

In [ ]:
def parse_date(s):
    try:
        return dparser.parse(s, default=datetime(2000, 1, 1), dayfirst=True)
    except Exception:
        return None

def extract_info(text):
    info = {}
    t = text.replace('\n', ' ')
    m = re.search(r'(\d+\s?(?:mg|mcg|g|ml|iu))', t, re.I)
    info['strength'] = m.group(1) if m else None
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    caps = [l for l in lines if l.isupper() and len(l) > 3]
    info['drug_name'] = caps[0].title() if caps else (lines[0] if lines else None)
    m = re.search(r'(?:dosage|dose|directions?)\s*[:\-]?\s*(.+?)(?:batch|mfg|exp|store|warning|$)', t, re.I)
    info['dosage'] = m.group(1).strip() if m else None
    m = re.search(r'batch\s*(?:no|number|#)?\s*[:.\-]?\s*([A-Z0-9]+)', t, re.I)
    info['batch_no'] = m.group(1) if m else None
    m = re.search(r'(?:mfg|manufactur\w*)\s*(?:date)?\s*[:.\-]?\s*(\d{1,2}[/\-]\d{2,4}|\w{3,9}\.?\s*\d{4})', t, re.I)
    info['mfg_date'] = m.group(1) if m else None
    m = re.search(r'(?:exp\w*)\s*(?:date)?\s*[:.\-]?\s*(\d{1,2}[/\-]\d{2,4}|\w{3,9}\.?\s*\d{4})', t, re.I)
    info['exp_date'] = m.group(1) if m else None
    m = re.search(r'(warning\s*[:\-]?\s*.+?\.|keep out of reach of children)', t, re.I)
    info['warning'] = m.group(1).strip() if m else None
    return info

info = extract_info(full_text)
for k, v in info.items():
    print(f'{k:10s}: {v}')

### 1.5 Cognitive assistant: reasoning + speech
The assistant checks whether the medicine is expired and produces a spoken, easy to understand summary (useful for visually impaired or elderly users).

In [ ]:
def check_expiry(exp_str):
    d = parse_date(exp_str) if exp_str else None
    if d is None:
        return 'unknown', None
    # month/year expiry is valid until the end of that month
    end = (d.replace(day=28) + __import__('datetime').timedelta(days=4)).replace(day=1)
    return ('EXPIRED' if datetime.now() >= end else 'VALID'), d

def build_message(info):
    status, d = check_expiry(info.get('exp_date'))
    parts = []
    if info.get('drug_name'): parts.append(f"This medicine is {info['drug_name']}")
    if info.get('strength'):  parts.append(f"strength {info['strength']}")
    if info.get('dosage'):    parts.append(f"Dosage instructions: {info['dosage']}")
    if info.get('exp_date'):  parts.append(f"Expiry date is {info['exp_date']}")
    if status == 'EXPIRED':   parts.append('Warning. This medicine has expired. Do not use it.')
    elif status == 'VALID':   parts.append('The medicine is within its expiry date.')
    else:                     parts.append('I could not read the expiry date. Please check the bottle manually.')
    if info.get('warning'):   parts.append(info['warning'])
    return '. '.join(parts)

message = build_message(info)
print('ASSISTANT:', message)
tts = gTTS(message, lang='en')
tts.save('medicine_summary.mp3')
display(Audio('medicine_summary.mp3'))

### 1.6 Batch evaluation on a dataset (optional)
If you downloaded a dataset with ground truth drug names, put them in a dict `ground_truth = {'filename.jpg': 'Paracetamol', ...}` to compute a simple name-extraction accuracy.

In [ ]:
from difflib import SequenceMatcher

ground_truth = {}   # e.g. {'img1.jpg': 'Paracetamol'}
rows = []
for p in image_paths[:20]:
    im = cv2.imread(p)
    if im is None: continue
    txt = '\n'.join(t for _, t, _ in run_ocr(im))
    inf = extract_info(txt)
    rows.append((os.path.basename(p), inf['drug_name'], inf['exp_date']))
    print(rows[-1])

if ground_truth:
    sims = [SequenceMatcher(None, (n or '').lower(), ground_truth[f].lower()).ratio()
            for f, n, _ in rows if f in ground_truth]
    print('Mean name similarity:', np.mean(sims))

---
# PART 2: Emotion-aware Cognitive Assistant using Facial Expression Detection

**Pipeline:** Image / webcam frame -> Face detection (Haar cascade) -> Emotion classification (CNN) -> Emotion-aware response from the assistant

Two approaches are given:
- **A. Pre-trained model (DeepFace)** for quick inference
- **B. Custom CNN trained on FER2013** (dataset from Kaggle: *msambare/fer2013*)

In [ ]:
!pip -q install deepface tf-keras

In [ ]:
import cv2, numpy as np, matplotlib.pyplot as plt
from deepface import DeepFace
from skimage import data

face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
EMOTIONS = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
print('Ready')

### 2.1 Emotion-aware dialogue policy
The assistant changes tone and suggestions depending on the detected emotion.

In [ ]:
RESPONSES = {
    'happy':    'You look happy! Great to see that. Want me to keep the good mood going with some upbeat music?',
    'sad':      'You seem a bit down. I am here for you. Would you like to talk, or listen to something calming?',
    'angry':    'I sense you are upset. Let us take a deep breath together. Shall I pause notifications for a while?',
    'fear':     'You seem worried. Everything will be fine. Can I help you with whatever is troubling you?',
    'surprise': 'Looks like something surprised you! Do you want to tell me about it?',
    'disgust':  'That did not look pleasant. Would you like me to change what we are doing?',
    'neutral':  'You seem calm. How can I help you today?'
}

def assistant_reply(emotion):
    return RESPONSES.get(emotion, RESPONSES['neutral'])

### 2.2 Approach A: Detect faces and emotions with a pre-trained model
Replace `image_bgr` with your own photo (e.g. `cv2.imread('my_face.jpg')`) or a webcam frame.

In [ ]:
def detect_emotions(image_bgr):
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(40, 40))
    out = []
    for (x, y, w, h) in faces:
        roi = image_bgr[y:y+h, x:x+w]
        res = DeepFace.analyze(roi, actions=['emotion'], enforce_detection=False, silent=True)
        res = res[0] if isinstance(res, list) else res
        out.append(((x, y, w, h), res['dominant_emotion'], res['emotion']))
    return out

# demo image with a face (from scikit-image); replace with your own photo / webcam capture
image_bgr = cv2.cvtColor(data.astronaut(), cv2.COLOR_RGB2BGR)

detections = detect_emotions(image_bgr)
vis = image_bgr.copy()
for (x, y, w, h), emo, scores in detections:
    cv2.rectangle(vis, (x, y), (x+w, y+h), (0, 255, 0), 2)
    cv2.putText(vis, emo, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
    print('Detected emotion:', emo)
    print('ASSISTANT:', assistant_reply(emo))
plt.figure(figsize=(6, 6)); plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)); plt.axis('off'); plt.show()

### 2.3 Webcam capture in Colab (optional)
Run this cell, take a photo, then the assistant responds to your emotion.

In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode

def take_photo(filename='photo.jpg', quality=0.9):
    js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = 'Capture';
      div.appendChild(capture);
      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});
      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();
      await new Promise((resolve) => capture.onclick = resolve);
      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
    display(js)
    data_url = eval_js('takePhoto({})'.format(quality))
    with open(filename, 'wb') as f:
        f.write(b64decode(data_url.split(',')[1]))
    return filename

# Uncomment to use the webcam:
# path = take_photo()
# frame = cv2.imread(path)
# for (x, y, w, h), emo, _ in detect_emotions(frame):
#     print(emo, '->', assistant_reply(emo))

### 2.4 Approach B: Train a custom CNN on FER2013
Download the dataset from Kaggle (`msambare/fer2013`). Expected structure: `fer2013/train/<emotion>/*.jpg` and `fer2013/test/<emotion>/*.jpg` (48x48 grayscale).

In [ ]:
# Download via Kaggle API (upload your kaggle.json first), or upload the dataset folder manually
# !pip -q install kaggle
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d msambare/fer2013 && unzip -q fer2013.zip -d fer2013

import os
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator

TRAIN_DIR, TEST_DIR = 'fer2013/train', 'fer2013/test'

if os.path.isdir(TRAIN_DIR):
    train_gen = ImageDataGenerator(rescale=1/255., rotation_range=15, zoom_range=0.1,
                                   horizontal_flip=True, validation_split=0.1)
    test_gen = ImageDataGenerator(rescale=1/255.)
    train_ds = train_gen.flow_from_directory(TRAIN_DIR, target_size=(48, 48), color_mode='grayscale',
                                             class_mode='categorical', batch_size=64, subset='training')
    val_ds = train_gen.flow_from_directory(TRAIN_DIR, target_size=(48, 48), color_mode='grayscale',
                                           class_mode='categorical', batch_size=64, subset='validation')
    test_ds = test_gen.flow_from_directory(TEST_DIR, target_size=(48, 48), color_mode='grayscale',
                                           class_mode='categorical', batch_size=64, shuffle=False)
    class_names = list(train_ds.class_indices.keys())
    print(class_names)
else:
    print('FER2013 not found. Download it to run Approach B (Approach A above already works).')

In [ ]:
def build_cnn(num_classes=7):
    m = models.Sequential([
        layers.Input((48, 48, 1)),
        layers.Conv2D(32, 3, padding='same', activation='relu'), layers.BatchNormalization(),
        layers.Conv2D(32, 3, padding='same', activation='relu'), layers.MaxPooling2D(), layers.Dropout(0.25),
        layers.Conv2D(64, 3, padding='same', activation='relu'), layers.BatchNormalization(),
        layers.Conv2D(64, 3, padding='same', activation='relu'), layers.MaxPooling2D(), layers.Dropout(0.25),
        layers.Conv2D(128, 3, padding='same', activation='relu'), layers.BatchNormalization(),
        layers.MaxPooling2D(), layers.Dropout(0.3),
        layers.Flatten(),
        layers.Dense(256, activation='relu'), layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='categorical_crossentropy', metrics=['accuracy'])
    return m

cnn = build_cnn()
cnn.summary()

In [ ]:
if os.path.isdir(TRAIN_DIR):
    cb = [tf.keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True),
          tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3)]
    history = cnn.fit(train_ds, validation_data=val_ds, epochs=30, callbacks=cb)
    cnn.save('emotion_cnn.keras')

    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1); plt.plot(history.history['accuracy'], label='train'); plt.plot(history.history['val_accuracy'], label='val'); plt.title('Accuracy'); plt.legend()
    plt.subplot(1, 2, 2); plt.plot(history.history['loss'], label='train'); plt.plot(history.history['val_loss'], label='val'); plt.title('Loss'); plt.legend()
    plt.show()

    loss, acc = cnn.evaluate(test_ds)
    print(f'Test accuracy: {acc:.3f}')

In [ ]:
if os.path.isdir(TRAIN_DIR):
    from sklearn.metrics import classification_report, confusion_matrix
    import seaborn as sns
    probs = cnn.predict(test_ds)
    y_pred = probs.argmax(1)
    y_true = test_ds.classes
    print(classification_report(y_true, y_pred, target_names=class_names))
    plt.figure(figsize=(7, 6))
    sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names, cmap='Blues')
    plt.xlabel('Predicted'); plt.ylabel('True'); plt.show()

### 2.5 Emotion-aware assistant using the trained CNN

In [ ]:
def emotion_assistant_cnn(image_bgr, model, labels):
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.1, 5, minSize=(40, 40))
    vis = image_bgr.copy()
    for (x, y, w, h) in faces:
        face = cv2.resize(gray[y:y+h, x:x+w], (48, 48)).astype('float32') / 255.0
        p = model.predict(face[None, ..., None], verbose=0)[0]
        emo = labels[int(p.argmax())]
        cv2.rectangle(vis, (x, y), (x+w, y+h), (255, 0, 0), 2)
        cv2.putText(vis, f'{emo} {p.max():.2f}', (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 2)
        print('Detected:', emo, '| ASSISTANT:', assistant_reply(emo))
    plt.figure(figsize=(6, 6)); plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)); plt.axis('off'); plt.show()

if os.path.isdir(TRAIN_DIR):
    emotion_assistant_cnn(image_bgr, cnn, class_names)

---
## Conclusion
- **Medicine label assistant:** OpenCV pre-processing + EasyOCR reliably reads printed labels; regex-based extraction pulls out drug name, strength, dosage, batch and expiry; the assistant flags expired medicine and reads the summary aloud, which helps visually impaired and elderly users.
- **Emotion-aware assistant:** Haar cascade finds faces, a CNN (pre-trained DeepFace or custom FER2013 model) classifies seven emotions, and the assistant adapts its reply to the user's emotional state.
- **Limitations / future work:** curved bottle text, glare and handwriting reduce OCR accuracy (could add text-detection models like CRAFT and layout-aware models); FER2013 is noisy and class-imbalanced (could use transfer learning with ResNet / MobileNet and multimodal cues such as voice).

**Submitted by:** Arshpreet Singh